In [ ]:
import jax.numpy as jnp
import numpy as np
import jax
from master.symmetry import *
from master.phase_retrieval import *
from master.obj import *
from master.noise import *
from master.mle import *
import master.device as device
from master.pies import *
from master.ll_plotting import *
from master.ptychography import *
from matplotlib.patches import Circle
from matplotlib.lines import Line2D
from master.mle_pty import *
from master.operators import *
from master.spotgen import *
from master.probes import *
from master import *
from tqdm import tqdm

from pty_defs import cameraman_object, create_ops_and_pupil

jax.config.update("jax_enable_x64", True)
cdtype = jnp.array(1j).dtype
gpu_idx, free_mb, gpu_count = device.setup_device(device_id=3)

In [ ]:
N_crop = 256
N_pad = 32
N_full = N_crop + N_pad * 2
NA_det = 0.5
NA_spot = 1.0

x_obj, obj_support = cameraman_object(N_crop, N_pad)

r_x, r_y, X, Y, R, k_0, k_x, k_y, K_X, K_Y, k, r_extent, k_extent = (
    create_physical_axes(x_obj, lambda_0=1.0, pixel_size=1.0, NA=1.0)
)

A, A_dagger, F_pupil, pupil_support = create_ops_and_pupil(
    N_crop, N_pad, k, k_0, NA_det
)
print(r_extent, k_extent)

yhat = A(x_obj)
plot_intensity_phase(x_obj, extent=r_extent)
plot_intensity_phase(yhat, extent=k_extent, use_log=True)
plot_intensity_phase(A_dagger(yhat) * obj_support, extent=r_extent)
r_x, r_y, X, Y, R, k_0, k_x, k_y, K_X, K_Y, k, r_extent, k_extent

# 1. Multi-spot real-space ptychography with slight 2D scanning
Slight translations of the multi-spot pattern using a galvo-mirror should be possible. 

In [ ]:
min_x, max_x, min_y, max_y = r_extent

# Define multi-spot array properties like number of spots, NA of spot and spatial area it covers
L_x, L_y = 16, 16
# With detector and spot NA, a bound on the total synthetic aperture NA can be given
NA_syn_opt = NA_det + NA_spot
frac_x = 0.75
frac_y = 0.75

probe_type = "sobol"
probe_seed = 1776
M = 128

N_int_avg = 100

probes, f_ms, R_probes = construct_probe_fields(
    probe_type,
    probe_seed,
    NA_spot,
    L_x,
    L_y,
    frac_x,
    frac_y,
    r_extent,
    M,
    1.0,
    K_X,
    K_Y,
    k,
    k_0,
    cdtype,
)
object_probe_sum_unscaled = center_crop(
    jnp.sum(jnp.abs(probes * x_obj) ** 2, axis=0), N_crop
)
mean_unscaled = jnp.sum(object_probe_sum_unscaled)
N_photons = N_crop**2 * N_int_avg / (mean_unscaled)
probes, f_ms, R_probes = construct_probe_fields(
    probe_type,
    probe_seed,
    NA_spot,
    L_x,
    L_y,
    frac_x,
    frac_y,
    r_extent,
    M,
    N_photons,
    K_X,
    K_Y,
    k,
    k_0,
    cdtype,
)
ys = A(x_obj * probes)
probed = analyse_probe(probes, x_obj, r_extent, single_plot=True)
object_probe_sum = center_crop(jnp.sum(jnp.abs(probes * x_obj) ** 2, axis=0), N_crop)
plot(object_probe_sum)
print(jnp.mean(object_probe_sum))

# 2. Calculate spectral support of probe and detector together

In [ ]:
from math import ceil


@jit
def create_displaced_dirac(delta_k_x, delta_k_y, x, y):
    return jnp.exp(delta_k_x * 1.0j * x[None, :] + delta_k_y * 1.0j * y[:, None])


@jit
def create_per_multispot_array(k):
    k_x = 2 * jnp.pi / array_period_x
    k_y = 2 * jnp.pi / array_period_y
    print(k_0)

    i_bound = ceil((NA_spot * k_0 / 2) / array_period_x) + 1
    j_bound = ceil((NA_spot * k_0) / array_period_y) + 1
    total = jnp.zeros_like(k)
    for i in np.arange(-i_bound, i_bound + 1):
        print(i)
        for j in np.arange(-j_bound, j_bound + 1):
            delta_x = k_x * i
            delta_y = k_y * j
            if np.sqrt(delta_x**2 + delta_y**2) <= NA_spot * k_0:
                spot_spectrum = create_displaced_dirac(
                    delta_x,
                    delta_y,
                    r_x,
                    r_y,
                )
                total += fft2_shifted(spot_spectrum)
    # Use simple linearity of fft2_shifted to avoid calculating too many fft2s
    return total


array_bounds, array_period_x, array_period_y = calc_array_properties(
    L_x, L_y, frac_x, frac_y, r_extent
)
t = create_per_multispot_array(k)
circle = Circle(
    (0, 0),
    k_0 * NA_spot,
    fill=False,
    edgecolor="red",
    linewidth=2.2,
    linestyle="--",
)
plot_intensity_phase(t, extent=k_extent, label="P", circle=circle)

In [ ]:
base_probe = probes[0]
# For a finite probe, it is best to use the full synthetic aperture to ensure also the weakly estimable parts of the spectrum are captured
# For an infinite probe, the exact coverage as union of circles centered at probe deltas becomes the right model
spectral_support = build_spectral_support_analytical(k, k_0, NA_syn_opt)
plot_intensity_phase(ifft2_shifted(fft2_shifted(base_probe) * F_pupil), extent=r_extent)
plot_intensity_phase(spectral_support)

In [ ]:
base_probe = create_probes(jnp.array([[0.0], [0.0]]), f_ms, N_photons, cdtype)[0]
probe_spect = fft2_shifted(base_probe)

circle = Circle(
    (0, 0),
    k_0 * 0.5,
    fill=False,
    edgecolor="red",
    linewidth=2.2,
    linestyle="--",
)
plot_intensity_phase(probe_spect, extent=k_extent, label="P", circle=circle)

## 2.1 Plot approximate spectral support of measurement

In [ ]:
y = jnp.abs(probe_spect) ** 2
if jnp.iscomplexobj(y):
    y = jnp.abs(y)

fig, ax = plt.subplots(figsize=(6, 6), constrained_layout=True)

im = ax.imshow(np.asarray(y), extent=k_extent)
cbar = plt.colorbar(im, ax=ax, fraction=0.045, pad=0.02)

ax.add_patch(
    Circle(
        (0, 0),
        k_0 * NA_spot,
        fill=False,
        edgecolor="red",
        linewidth=2.2,
        linestyle="--",
    )
)

# Labels / legend
handles = [
    Line2D(
        [0],
        [0],
        color="red",
        lw=2.2,
        linestyle="--",
        label=rf"$NA_{{spot}}={NA_spot}$",
    ),
]

ax.legend(handles=handles, loc="upper right", fontsize=11)
ax.set_aspect("equal")

plt.show()

In [ ]:
plot(spectral_support)

In [ ]:
plot_intensity_phase(
    fft2_shifted(x_obj) * spectral_support, extent=k_extent, use_log=True
)
P_spectral_support = lambda x: ifft2_shifted(fft2_shifted(x) * spectral_support)
x_obj_est = P_spectral_support(x_obj)
plot_intensity_phase(x_obj_est, extent=r_extent)

In [ ]:
x_est = pie(
    ys,
    probes,
    gamma=0.75,
    N_iter=400,
    beta_reg=0.1,
    # obj_support=1,
    A=A,
    A_dagger=A_dagger,
)
canonicalize_est = lambda x_est: np.asarray(
    choose_representative_broadcast(P_spectral_support(x_est), x_obj_est) * probed
)
compare_estimate(
    canonicalize_est(x_est), canonicalize_est(x_obj), N_compare_crop=N_crop
)